# Pertemuan 3 — Data Cleaning

Nama: Novi Shandi  
NIM: 240401010291

Praktikum Data Cleaning menggunakan Pandas dan NumPy

In [55]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests

## STEP 1 — Load Dataset

Dataset housing_dirty.csv dimuat ke dalam DataFrame Pandas untuk proses eksplorasi dan pembersihan data.

In [56]:
df = pd.read_csv('housing_dirty.csv')

print("Shape awal dataset:")
print(df.shape)

df.head()

Shape awal dataset:
(130, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


## STEP 2 — Eksplorasi Awal

Dilakukan eksplorasi awal untuk:
- melihat tipe data
- mengecek missing values
- memahami struktur dataset

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [58]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [59]:
df.isnull().sum()

,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


## STEP 3 — Menghapus Data Duplikat

Data duplikat dapat menyebabkan hasil analisis menjadi bias karena data yang sama dihitung lebih dari satu kali.

In [60]:
print("Jumlah duplikat sebelum dihapus:")
print(df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Jumlah duplikat setelah dihapus:")
print(df.duplicated().sum())

Jumlah duplikat sebelum dihapus:
0
Jumlah duplikat setelah dihapus:
0


## STEP 4 — Normalisasi String

Dilakukan normalisasi agar penulisan data menjadi konsisten.

Contoh:
- 'jakarta' → 'Jakarta'
- 'BAIK' → 'baik'

In [61]:
df['kota'] = df['kota'].str.strip().str.title()

df['kondisi'] = df['kondisi'].str.strip().str.lower()

df[['kota', 'kondisi']].head()

,kota,kondisi
0,Jogja,baik
1,Medan,bagus
2,Depok,baik
3,Ygy,baik
4,Medan,sedang


## STEP 5 — Menangani Missing Values

Missing values ditangani menggunakan:
- median untuk data numerik
- modus untuk data kategorik

Median dipilih karena lebih tahan terhadap outlier dibanding mean.

In [62]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())

df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

In [63]:
df.isnull().sum()

,0
id,0
luas_m2,0
harga_juta,0
kota,0
kamar,0
tahun_bangun,0
kondisi,0


## STEP 6 — Menangani Outlier

Outlier adalah nilai ekstrem yang dapat mempengaruhi hasil analisis.

Metode yang digunakan:
- IQR Fence
- clip() untuk membatasi nilai ekstrem

In [64]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

print("Outlier berhasil ditangani")

Outlier berhasil ditangani


Outlier ditangani menggunakan metode IQR Fence dengan fungsi clip().

Metode ini membatasi nilai ekstrem agar tidak terlalu mempengaruhi distribusi data tanpa harus menghapus data secara keseluruhan.

Pendekatan ini lebih aman dibanding langsung menghapus baris karena jumlah data tetap dipertahankan.

## STEP 7 — Validasi Dataset

Dilakukan validasi akhir untuk memastikan:
- tidak ada missing values
- tidak ada data duplikat

In [65]:
print("Total missing values:")
print(df.isnull().sum().sum())

print("Total duplikat:")
print(df.duplicated().sum())

Total missing values:
0
Total duplikat:
0


## STEP 5 — Menangani Missing Values

Missing values ditangani menggunakan:
- median untuk data numerik
- modus untuk data kategorik

Median dipilih karena lebih tahan terhadap outlier dibanding mean.

In [66]:
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())

df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

In [67]:
df.isnull().sum()

,0
id,0
luas_m2,0
harga_juta,0
kota,0
kamar,0
tahun_bangun,0
kondisi,0


## STEP 6 — Menangani Outlier

Outlier adalah nilai ekstrem yang dapat mempengaruhi hasil analisis.

Metode yang digunakan:
- IQR Fence
- clip() untuk membatasi nilai ekstrem

In [68]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

print("Outlier berhasil ditangani")

Outlier berhasil ditangani


## STEP 7 — Validasi Dataset

Dilakukan validasi akhir untuk memastikan:
- tidak ada missing values
- tidak ada data duplikat

In [69]:
print("Total missing values:")
print(df.isnull().sum().sum())

print("Total duplikat:")
print(df.duplicated().sum())

Total missing values:
0
Total duplikat:
0


## STEP 8 — Export Dataset Bersih

Dataset yang telah dibersihkan disimpan menjadi file CSV baru.

In [70]:
df.to_csv('housing_clean.csv', index=False)

print("Dataset berhasil disimpan")

Dataset berhasil disimpan


## STEP 9 — Mengakses REST API

Data diambil dari API publik JSONPlaceholder menggunakan library requests.

In [71]:
url = "https://jsonplaceholder.typicode.com/users"

response = requests.get(url, timeout=10)

if response.status_code == 200:

    data = response.json()

    api_df = pd.json_normalize(data)

    api_df[['id', 'name', 'email', 'address.city']]

else:
    print("Error:", response.status_code)

In [72]:
print(api_df.head())

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburgh

## Interpretasi API

Data berhasil diambil dari REST API dan dikonversi menjadi DataFrame Pandas.

Informasi yang diperoleh meliputi:
- ID pengguna
- Nama
- Email
- Kota

Hal ini menunjukkan bahwa Pandas dapat digunakan untuk mengolah data dari sumber online secara langsung.

# Kesimpulan

Pada praktikum ini dilakukan proses data cleaning terhadap dataset housing_dirty.csv.

Tahapan yang dilakukan:
- eksplorasi data
- deteksi missing values
- penghapusan duplikat
- normalisasi string
- imputasi data kosong
- penanganan outlier
- export dataset bersih
- akses REST API

Hasil akhir menunjukkan dataset berhasil dibersihkan dan siap digunakan untuk analisis lebih lanjut maupun machine learning.

Data cleaning merupakan tahap penting sebelum analisis data dan machine learning dilakukan.

Dataset yang bersih akan menghasilkan analisis yang lebih akurat, konsisten, dan dapat dipercaya.